Mit Renku, Binder oder Colab kann das Jupyter-Notebook interaktiv im Browser gestartet werden:

[![Python - renku](https://renkulab.io/renku-badge.svg)](https://renkulab.io/p/opendatazurich/github-jupyter/sessions/01M3PPQJDDQZ4YD7CD7DZ5S5ER/start?GITHUB_FILEPATH=opendatazurich/opendatazurich.github.io/blob/master/geoportal/Geoportal-Beispiele.ipynb)

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/opendatazurich/opendatazurich.github.io/master?filepath=geoportal/Geoportal-Beispiele.ipynb)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opendatazurich/opendatazurich.github.io/blob/master/geoportal/Geoportal-Beispiele.ipynb)

# Daten via WFS beziehen

Ein WFS-Dienst liefert Vektor-Geodaten. Überlicherweise sind bei einem WFS mehrere Layer enthalten, welche jeweils unterschiedliche Daten enthalten können. [Esri hat eine gute Anleitung](https://enterprise.arcgis.com/de/server/10.3/publish-services/linux/communicating-with-a-wfs-service-in-a-web-browser.htm), wie mit einem WFS-Server kommuniziert werden kann.

In diesem Notebook wird gezeigt, wie Daten via WFS abgefragt werden können (z.B. als GeoJSON) um diese dann weiterzuverarbeiten. Die Daten können dabei auf einer Karte dargestellt werden, oder auch als tabellarische Daten aufbereitet werden.
Zum Schluss gibt es noch einige Hinweise, wie Daten via dem WFS-Dienst gefiltert werden können (sowohl geografisch wie auch auf Attributebene).

Hinweis: die URL zum WFS Dienst finden man via Geoportal, welches auf dem OGD-Katalog auf den Geo-Datensätzen verlinkt ist.

In [ ]:
%pip install geopandas requests folium

In [ ]:
import xml.etree.ElementTree as ET

import geopandas
import requests
import folium

In [ ]:
wfs_url = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Statistische_Quartiere"

## GetCapabilities

In [ ]:
# GetCapabilities zeigt die möglichen Anfragen an einen WFS-Server
r = requests.get(wfs_url, params={
    'service': 'WFS',
    'version': '1.0.0',
    'request': 'GetCapabilities'
})
r.content

In [ ]:
# XML parsen und die Layer-Informationen extrahieren
root = ET.fromstring(r.content)
namespaces = {
    'wfs': 'http://www.opengis.net/wfs'
}
layers = {}
for feature_type in root.findall('wfs:FeatureTypeList/wfs:FeatureType', namespaces):
    layers[feature_type.find('wfs:Name', namespaces).text] = {
        'srs': feature_type.find('wfs:SRS', namespaces).text,
    }

layers

Der Layer **adm_statistische_quartiere_map** ist der kartogratisch aufbereitete Layer. Die Details zu den einzelnen Layern sind auf dem OGD-Katalog beschrieben oder direkt auf geocat.ch.

Beispiel: https://www.geocat.ch/geonetwork/srv/ger/md.viewer#/full_view/fd1a94fe-4bd4-4a40-99af-8b859dfe82a7
![](geocat_layer_doc.png)

In [ ]:
# Prüfen, welche Formate GetFeature bietet
formats = root.find('wfs:Capability/wfs:Request/wfs:GetFeature/wfs:ResultFormat', namespaces)
for child in formats:
    print(child.tag)

## Daten via GetFeature als GeoJSON laden

In [ ]:
# Yay, GeoJSON!
# Daten als GeoJSON holen
layer = 'adm_statistische_quartiere_map'

r = requests.get(wfs_url, params={
    'service': 'WFS',
    'version': '1.0.0',
    'request': 'GetFeature',
    'typename': layer,
    'outputFormat': 'GeoJSON'
})
stat_quarter_geo = r.json()
stat_quarter_geo

### Daten in GeoPandas als Tabelle

In [ ]:
# load GeoJSON in geopandas
srs = layers[layer]['srs']
data = geopandas.GeoDataFrame.from_features(stat_quarter_geo, crs={'init': srs})
data

Link zum GeoJSON: https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Statistische_Quartiere?service=WFS&version=1.0.0&request=GetFeature&outputFormat=GeoJSON&typename=adm_statistische_quartiere_map

### Pagination

Falls ein WFS sehr viele Daten zurückliefert, ist es auch möglich mit Pagination jeweils nur einen Teil der Daten zu beziehen. Dazu dienen die beiden Parameter `startIndex` und `maxFeatures`.

Um Informationen zu den Resultaten zu bekommen, kann ein GetFeature-Request mit dem `resultType` **hits** gemacht werden:

In [ ]:
# Start bei Index 5 (`startIndex=5`), d.h. 0-4 werden ausgelassen und holen 1 Feature (`maxFeatures=1`)
r = requests.get(wfs_url, params={
    'service': 'WFS',
    'version': '1.0.0',
    'request': 'GetFeature',
    'typename': layer,
    'resultType': 'hits'
})
r.content

In [ ]:
# Start bei Index 5 (`startIndex=5`), d.h. 0-4 werden ausgelassen und holen 1 Feature (`maxFeatures=1`)
r = requests.get(wfs_url, params={
    'service': 'WFS',
    'version': '1.0.0',
    'request': 'GetFeature',
    'typename': layer,
    'outputFormat': 'GeoJSON',
    'startIndex': 5,
    'maxFeatures': 1
})
first_page = r.json()
first_data = geopandas.GeoDataFrame.from_features(first_page, crs={'init': srs})
first_data

### Daten in Karte integrierten

In [ ]:
# Basiskarte mit GeoJSON layer
m = folium.Map(location=[47.38, 8.53], zoom_start=13, tiles=None)
folium.raster_layers.WmsTileLayer(
    url='https://www.ogd.stadt-zuerich.ch/wms/geoportal/Basiskarte_Zuerich_Raster_Grau',
    layers='Basiskarte_Zuerich_Raster_Grau',
    name='Zürich - Basiskarte',
    fmt='image/png',
    overlay=False,
    control=False,
    autoZindex=False,
).add_to(m)
folium.features.GeoJson(stat_quarter_geo).add_to(m)
m

## Daten filtern

### Daten mit Attributen filtern

#### OGC Filter
Um Attribute zu filtern, kann man sogenannte _OGC Filter_ erstellen, welche als XML formuliert werden müssen.
Diese können dann über den `Filter` Parameter dem `GetFeature` Request mitgegeben werden, so dass die Daten dann auf dem Server gefiltert werden.

In [ ]:
# Filter für Kreis 7
filter_xml = """<?xml version="1.0"?>
<Filter xmlns="http://www.opengis.net/ogc" xmlns:gml="http://www.opengis.net/gml"> 
    <PropertyIsEqualTo>
        <PropertyName>knr</PropertyName>
        <Literal>7</Literal>
    </PropertyIsEqualTo>
</Filter>
"""

r = requests.get(wfs_url, params={
    'service': 'WFS',
    'version': '1.0.0',
    'request': 'GetFeature',
    'typename': layer,
    'outputFormat': 'GeoJSON',
    'Filter': filter_xml
})
filtered_geo = r.json()
filtered_data = geopandas.GeoDataFrame.from_features(filtered_geo, crs={'init': srs})
filtered_data

#### EXP_FILTER (QGIS WFS Server)

Zusätzlich unterstützt der verwendete QGIS WFS Server den Hersteller-spezifischen Parameter `EXP_FILTER`.

Mögliche Operatoren:

* field = 10
* field_a > 10
* field_b > 10 AND field_c <= 7
* field_d LIKE '%H'

In [ ]:
# Filter für Quartier LIKE 'H%'

r = requests.get(wfs_url, params={
    'service': 'WFS',
    'version': '1.0.0',
    'request': 'GetFeature',
    'typename': layer,
    'outputFormat': 'GeoJSON',
    'EXP_FILTER': "qname LIKE 'H%'"
})
like_geo = r.json()
like_data = geopandas.GeoDataFrame.from_features(like_geo, crs={'init': srs})
like_data

### Daten geografisch filtern

Die Daten vom WFS lassen sich auch geografisch filtern. Wir können z.B. Polygone oder Bounding Boxes verwenden.

Im folgenden Beispiel holen wir uns alle Kirsch-Bäume aus dem Baumkataster, welche im Kreis 7 sind.

In [ ]:
kreis7_geo_filter = """<?xml version="1.0"?>
<ogc:Filter xmlns:ogc="http://www.opengis.net/ogc">
    <ogc:And>
    <ogc:Intersects>
        <ogc:PropertyName>geometry</ogc:PropertyName>
        <gml:Box srsName="EPSG:2056">
            <gml:coordinates cs="," ts=" ">2684454.0423,1245344.4639 2687039.4349,1247714.1562</gml:coordinates>
        </gml:Box>
    </ogc:Intersects>
    <ogc:PropertyIsEqualTo>
        <ogc:PropertyName>baumgattunglat</ogc:PropertyName>
        <ogc:Literal>Prunus</ogc:Literal>
    </ogc:PropertyIsEqualTo>
    </ogc:And>
</ogc:Filter>
"""

baumkataster_wfs = 'https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Baumkataster'
baum_layer = 'baumkataster_baumstandorte'
r = requests.get(baumkataster_wfs, params={
    'service': 'WFS',
    'version': '1.0.0',
    'request': 'GetFeature',
    'typename': baum_layer,
    'outputFormat': 'GeoJSON',
    'Filter': kreis7_geo_filter
})
tree_geo = r.json()
tree_data = geopandas.GeoDataFrame.from_features(tree_geo, crs={'init': srs})
tree_data

In [ ]:
tree_map = folium.Map(location=[47.38, 8.53], zoom_start=13, tiles=None)
folium.raster_layers.WmsTileLayer(
    url='https://www.ogd.stadt-zuerich.ch/wms/geoportal/Basiskarte_Zuerich_Raster_Grau',
    layers='Basiskarte_Zuerich_Raster_Grau',
    name='Zürich - Basiskarte',
    fmt='image/png',
    overlay=False,
    control=False,
    autoZindex=False,
).add_to(tree_map)
folium.features.GeoJson(tree_geo).add_to(tree_map)
tree_map